# Ch 06 · Many Worlds: The Batched Data Layout

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/master/dpdk-research/mujoco-warp-course/notebooks/06-batched-data-layout.ipynb)

Companion notebook for the chapter page `chapters/06-batched-data-layout.html` ([page](../chapters/06-batched-data-layout.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

Every code cell below is the same file as the matching block on the chapter page (`examples/06-batched-data-layout/`). On a GPU the budgets grow automatically (`ON_GPU`); on a CPU they stay small. Timings printed here are **your** numbers.

Sections below follow the chapter page: each heading names the page section (§N), and the examples appear in the same order and with the same numbers as on the page (file names keep their exNN prefix).

## §1. The nworld-first rule
Every `mjw.Data` array starts with `nworld`. Vector dtypes (`vec3`, `quat`, `mat33`, `spatial_vector`) add trailing dimensions only in the NumPy view (page §2).

In [ ]:
# Example 6.1 - ex01_shape_table.py
"""Example 6.1 - the nworld-first rule: print the shape table of mjw.Data.

Every Data array starts with an nworld dimension (one row per world). Vector dtypes
(vec3, quat, mat33, spatial_vector) add trailing dimensions only in the NumPy view.
Run: python ex01_shape_table.py
"""
import dataclasses

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

NWORLD = 4  # shapes only: a small nworld keeps the table readable on CPU and GPU

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
print(f"nq={mjm.nq} nv={mjm.nv} nu={mjm.nu} nbody={mjm.nbody} ngeom={mjm.ngeom} nworld={d.nworld}")
print(f"{'field':<16}{'warp shape':<14}{'dtype':<17}{'numpy shape':<14}")

scalars, rows, total, not_world = [], 0, 0, []
for f in dataclasses.fields(d):
    arr = getattr(d, f.name)
    if not isinstance(arr, wp.array):
        continue
    total += 1
    if arr.size and arr.shape[0] != d.nworld:  # (empty workspaces are skipped)
        not_world.append(f.name)
    if arr.shape == (d.nworld,) and arr.dtype in (wp.int32, wp.float32):
        scalars.append(f.name)  # one number per world: summarized below
        continue
    if arr.size == 0 or rows >= 25:
        continue  # skip empty arrays (this model has no mocap, act, ...)
    np_shape = arr.numpy().shape  # vector dtypes append their length here
    print(f"{f.name:<16}{str(arr.shape):<14}{arr.dtype.__name__:<17}{str(np_shape):<14}")
    rows += 1
print(f"(nworld,) scalars: {', '.join(scalars)}")
print(f"{total} top-level arrays in Data; non-empty ones without a leading nworld: {not_world}")

print("-- pooled and per-world sub-structs --")
for name in ("contact.pos", "contact.worldid", "efc.force", "efc.J", "nacon", "nefc"):
    obj = d
    for part in name.split("."):
        obj = getattr(obj, part)
    print(f"{name:<16}{str(obj.shape):<14}{obj.dtype.__name__:<17}")
print(f"naconmax={d.naconmax} (nconmax 16 x nworld {d.nworld}), njmax={d.njmax} per world")

## §3. From shape to bytes
Row-major layout: `offset = ((w * nbody + b) * 3 + k) * 4`. A `dim=(nworld, nbody)` launch numbers threads in the same order, so a warp reads contiguous bytes.

In [ ]:
# Example 6.2 - ex08_addresses.py
"""Example 6.2 - from shape to bytes: where d.xpos[w, b][k] lives in memory.

Warp arrays are row-major (C order). d.xpos has Warp shape (nworld, nbody) of vec3 (12 bytes),
so its byte offset is ((w * nbody + b) * 3 + k) * 4. We check the formula against .strides
and against the flat buffer, then list which bytes one 32-thread warp touches.
Run: python ex08_addresses.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

NWORLD = 8  # 8 worlds x 4 bodies = 32 (w, b) pairs = exactly one warp

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
nbody = mjm.nbody
# label every float32 word with its own index, so a value tells us where it was read from
d.xpos.assign(np.arange(NWORLD * nbody * 3, dtype=np.float32).reshape(NWORLD, nbody, 3))

print("d.xpos shape", d.xpos.shape, "strides (bytes)", d.xpos.strides,
      "contiguous", d.xpos.is_contiguous)
print("m.body_mass shape", m.body_mass.shape, "strides", m.body_mass.strides)

full = d.xpos.numpy()                      # (nworld, nbody, 3) view of the same bytes
print("  w  b  k   strides offset   formula   word label = offset/4")
for w, b, k in [(0, 0, 0), (0, 1, 0), (0, 1, 2), (1, 0, 0), (2, 3, 1), (7, 3, 2)]:
    off = w * d.xpos.strides[0] + b * d.xpos.strides[1] + k * 4
    formula = ((w * nbody + b) * 3 + k) * 4
    label = int(full[w, b, k])
    print(f"  {w}  {b}  {k}   {off:14d}   {formula:7d}   {label:5d} {label * 4 == off}")

# Warp unravels a 2-D launch index with the LAST dimension fastest (row-major):
# linear thread t -> (worldid, bodyid) = (t // nbody, t % nbody)
t = np.arange(32)                          # one warp = 32 consecutive threads
w_t, b_t = t // nbody, t % nbody
addr = (w_t * nbody + b_t) * 12
print(f"dim=(nworld, nbody): threads 0..31 = worlds {w_t.min()}..{w_t.max()},"
      f" bytes {addr.min()}..{addr.max() + 11}, contiguous: {bool((np.diff(addr) == 12).all())}")
addr_w = np.arange(NWORLD) * d.xpos.strides[0] + 1 * d.xpos.strides[1]
print("dim=nworld reading xpos[w, 1]: byte offsets", addr_w.tolist(), "(stride nbody*12)")

## §4. Writing per-world inputs in place
Build a full `(nworld, ...)` array and copy it *into* the device array with `.assign`. Then `forward` updates kinematics for all worlds.

In [ ]:
# Example 6.3 - ex03_per_world_qpos.py
"""Example 6.3 - a different hinge angle in every world, written in place with .assign.

Each row of d.qpos is one world. We build the full (nworld, nq) array on the host, change
column 14 (the arm hinge), copy it into the existing device array, and run forward to
update the kinematics. The arm tip is computed from xpos and xmat of the arm body.
Run: python ex03_per_world_qpos.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)

hinge_adr = mjm.jnt_qposadr[mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_JOINT, "hinge")]
arm = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_BODY, "arm")
angles = np.linspace(-np.pi / 2, np.pi / 2, NWORLD)

qpos = np.tile(mjm.qpos0, (NWORLD, 1))      # (nworld, nq), every world starts at qpos0
qpos[:, hinge_adr] = angles                  # one angle per world
d.qpos.assign(qpos)                          # copy into the SAME device array (in place)
mjw.forward(m, d)                            # kinematics: xpos, xmat, ... for all worlds

xpos = d.xpos.numpy()[:, arm]                # (nworld, 3): arm body origin per world
xmat = d.xmat.numpy()[:, arm]                # (nworld, 3, 3)
tip = xpos + xmat @ np.array([0.5, 0.0, 0.0])  # capsule end in the body frame -> world
print("hinge qpos address:", hinge_adr, "| arm body id:", arm)
print(" world  angle[rad]  tip x    tip z   formula z")
for w in range(min(NWORLD, 8)):
    z_formula = 1.0 - 0.5 * np.sin(angles[w])  # rotation about +y moves the tip down for q > 0
    print(f"{w:5d}  {angles[w]:9.4f}  {tip[w, 0]:7.4f}  {tip[w, 2]:7.4f}  {z_formula:8.4f}")
err = float(np.abs(tip[:, 2] - (1 - 0.5 * np.sin(angles))).max())
print(f"max |tip z - formula| over {NWORLD} worlds: {err:.1e}  (float32 on device)")

`d.ctrl` is `(nworld, nu)`: one control per world, set once, applied every step.

In [ ]:
# Example 6.4 - ex04_per_world_ctrl.py
"""Example 6.4 - a different control in every world: d.ctrl is (nworld, nu).

World w gets a motor torque in [-1, 1]. After 100 steps the hinge angles spread out,
while the ball and the box (same in every world) land identically.
Run: python ex04_per_world_ctrl.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NSTEP = 100

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)

ctrl = np.linspace(-1.0, 1.0, NWORLD, dtype=np.float32).reshape(NWORLD, mjm.nu)  # (nworld, nu)
d.ctrl.assign(ctrl)            # set once; the motor applies it on every step
for _ in range(NSTEP):
    mjw.step(m, d)
wp.synchronize()

qpos = d.qpos.numpy()          # one host read at the end, not inside the loop
print("ctrl shape:", d.ctrl.shape, "| qpos shape:", d.qpos.shape)
print("ctrl per world:      ", ctrl[:8, 0])
print("hinge qpos per world:", qpos[:8, 14])
print("ball z per world:    ", qpos[:8, 2])
print(f"hinge spread (max - min): {qpos[:, 14].max() - qpos[:, 14].min():.4f} rad")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

`fill_`, `zero_`, `wp.copy`, row views, and the trap: `assign`/`wp.copy` only refuse a source that is *larger* than the destination.

In [ ]:
# Example 6.5 - ex05_fill_copy_zero.py
"""Example 6.5 - in-place writes: fill_, zero_, wp.copy, row views, and a size trap.

All of these change the contents of an existing device array and keep its identity
(same object, same device pointer), which is what kernels and CUDA graphs rely on.
Run: python ex05_fill_copy_zero.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

NWORLD = 4  # small on purpose: we print whole arrays

mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)
ptr0 = d.qvel.ptr

# 1. fill_: one value everywhere (all worlds, all dofs)
d.qvel.fill_(0.5)
print("after fill_(0.5):\n", d.qvel.numpy())

# 2. a row view d.qvel[w] shares memory with world w only
d.qvel.zero_()
d.qvel[2].fill_(1.0)          # world 2 only: indexing one row returns a view, not a copy
print("after zero_() and qvel[2].fill_(1.0):\n", d.qvel.numpy())

# 3. wp.copy from another device array (same shape and dtype): one world gets an upward kick
kick = np.zeros((NWORLD, mjm.nv), dtype=np.float32)
kick[1, 2] = 3.0                                   # world 1, dof 2 = linear z velocity
wp.copy(d.qvel, wp.array(kick, dtype=float))
print("after wp.copy(d.qvel, kick):\n", d.qvel.numpy())

# 4. round trip: host -> device -> step -> host
for _ in range(10):
    mjw.step(m, d)
print("ball z after 10 steps:", d.qpos.numpy()[:, 2])
print("same object and pointer:", d.qvel.ptr == ptr0)

# 5. trap: assign/copy check only that the source is not LARGER (a flat byte copy)
d.qvel.zero_()
d.qvel.assign(np.ones((NWORLD - 1, mjm.nv)))      # one world short: no error!
print("assign with (nworld-1, nv) rows -> silently fills a prefix:\n", d.qvel.numpy())
try:
    d.qvel.assign(np.ones((NWORLD + 1, mjm.nv)))
except RuntimeError as e:
    print("one world too many ->", type(e).__name__ + ":", str(e)[:58] + "...")

## §5. Replace vs assign
`d.qpos.assign(x)` keeps the array (and its device pointer); `d.qpos = wp.array(x)` creates a new one. Anything holding the old array (your variables, a captured CUDA graph in Ch 24) keeps using the old memory.

In [ ]:
# Example 6.6 - ex06_replace_vs_assign.py
"""Example 6.6 - replacing an array is not the same as writing into it.

d.qpos.assign(x) keeps the array object (and its device pointer) and changes its contents.
d.qpos = wp.array(x) puts a NEW array on the Data object; anything that still holds the old
one (your own variables, a captured CUDA graph, Ch 24) keeps using the old memory.
Run: python ex06_replace_vs_assign.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

NWORLD = 4

mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)

old = d.qpos                       # e.g. a reference your controller or a graph holds
ptr = d.qpos.ptr
q = np.tile(mjm.qpos0, (NWORLD, 1)).astype(np.float32)

# A. assign: same object, same pointer, new contents
q[:, 2] = 2.0
d.qpos.assign(q)
print("A assign : old is d.qpos =", old is d.qpos, "| same ptr =", d.qpos.ptr == ptr,
      "| old z =", old.numpy()[:, 2])

# B. replace: a new object with a new pointer; 'old' is now disconnected
q[:, 2] = 3.0
d.qpos = wp.array(q, dtype=float)
print("B replace: old is d.qpos =", old is d.qpos, "| same ptr =", d.qpos.ptr == ptr,
      "| old z =", old.numpy()[:, 2])

# the simulation reads whatever d.qpos is NOW; writes through 'old' are lost
old.fill_(0.0)                     # "reset" through the stale reference: no effect on d
mjw.step(m, d)
print("after step: d.qpos z =", d.qpos.numpy()[:, 2], "| old z =", old.numpy()[:, 2])

# C. replace with the wrong dtype: fails at the next kernel launch, not at assignment
d.qpos = wp.array(q.astype(np.float64))
try:
    mjw.step(m, d)
except RuntimeError as e:
    msg = str(e)
    print("C float64 replace ->", msg[: msg.index("expects")].strip(), "... expects float32")

## §6. Which Model fields can vary per world?
A field can vary per world if and only if its spec in `types.py` starts with `"*"`: it then has a batch dimension B (default 1). Topology (types, tree, addresses) has no `"*"`; a few integer selectors such as `geom_dataid` do.

In [ ]:
# Example 6.7 - ex02_model_batched.py
"""Example 6.7 - which Model fields can vary per world?

Numeric parameters have a leading batch dimension B (default 1, shared by all worlds).
Structural fields (types, ids, addresses) have no batch dimension: all worlds share topology.
Run: python ex02_model_batched.py
"""
import dataclasses

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(PLAYGROUND)
m = mjw.put_model(mjm)

for name, arr in [("body_mass", m.body_mass), ("geom_size", m.geom_size),
                  ("opt.gravity", m.opt.gravity), ("geom_type", m.geom_type),
                  ("body_parentid", m.body_parentid)]:
    print(f"{name:<14}{str(arr.shape):<9}{arr.dtype.__name__:<9}"
          f"batched: {getattr(arr, '_is_batched', False)}")
print("nq (Python int):", m.nq, type(m.nq).__name__)


def count(obj, prefix=""):
    """Count warp arrays in a (nested) dataclass and how many carry _is_batched."""
    n_arr = n_bat = 0
    names = []
    for f in dataclasses.fields(obj):
        val = getattr(obj, f.name, None)
        if dataclasses.is_dataclass(val):
            a, b, sub = count(val, prefix + f.name + ".")
            n_arr, n_bat, names = n_arr + a, n_bat + b, names + sub
        elif isinstance(val, wp.array):
            n_arr += 1
            if getattr(val, "_is_batched", False):
                n_bat += 1
                names.append(prefix + f.name)
    return n_arr, n_bat, names


n_arr, n_bat, names = count(m)
print(f"Model arrays: {n_arr}, batched ('*' spec): {n_bat}, structural: {n_arr - n_bat}")
print("first batched fields:", names[:8])


def star_specs(cls):
    """Fields whose declared spec (types.py::array) starts with '*'."""
    return sum(1 for f in dataclasses.fields(cls) if getattr(f.type, "shape", ())[:1] == ("*",))


print(f"'*' specs declared: Model {star_specs(mjw.Model)} + Option {star_specs(mjw.Option)}"
      f" = {star_specs(mjw.Model) + star_specs(mjw.Option)}")

# batch_sizes: allocate a per-world copy of chosen fields at put_model time
m4 = mjw.put_model(mjm, batch_sizes={"body_mass": 4, "geom_friction": 4})
print("with batch_sizes: body_mass", m4.body_mass.shape, "geom_friction", m4.geom_friction.shape)
print("body_mass rows identical:", bool((m4.body_mass.numpy() == m4.body_mass.numpy()[0]).all()))

## §7. Round robin, not blocks
World `w` reads row `w % B` of a batched Model field.

In [ ]:
# Example 6.8 - ex07_gravity_round_robin.py
"""Example 6.8 - batched Model fields are dealt round robin: world w reads row w % B.

m.opt.gravity has spec array("*", wp.vec3): any leading size B. With B = 2, even worlds get
the first gravity and odd worlds the second, NOT "first half / second half".
Run: python ex07_gravity_round_robin.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
g = np.array([0.0, 0.0, -9.81], dtype=np.float32)


def vz_after(nworld, gravity_rows, nstep=5):
    """Ball z velocity per world after nstep steps from rest, for a gravity batch."""
    m.opt.gravity = wp.array(gravity_rows, dtype=wp.vec3)   # replace: B = len(gravity_rows)
    d = mjw.make_data(mjm, nworld=nworld, nconmax=4, njmax=16)
    for _ in range(nstep):
        mjw.step(m, d)
    return d.qvel.numpy()[:, 2]


print("default gravity shape:", m.opt.gravity.shape, "-> every world reads row 0")
vz = vz_after(4, [g, -g])
print("B=2, nworld=4: qvel z =", vz, "(5 steps x 0.002 s x 9.81 = 0.0981)")
print("  row used per world  =", [w % 2 for w in range(4)])

rows = [g, 0.5 * g, 0.0 * g]                         # B = 3 with nworld = 8
vz = vz_after(8, rows)
print("B=3, nworld=8: qvel z =", vz)
print("  row used per world  =", [w % 3 for w in range(8)])

per_world = [g * (1 + 0.25 * w) for w in range(4)]   # B = nworld: one row per world
print("B=nworld=4:    qvel z =", vz_after(4, per_world))
# a replaced array is a plain wp.array: put_model's mark_batched tag is not carried over
print("replaced gravity shape:", m.opt.gravity.shape,
      "| _is_batched tag:", getattr(m.opt.gravity, "_is_batched", False))

## §9. The spec strings, and the running project
The next cell writes `my_env.py` (the course's helper module, Ch 6 snapshot: `make_worlds`). Example 6.9 imports it and prints each field's docstring shape, declared spec and real shapes.

In [ ]:
%%writefile my_env.py
"""my_env.py, snapshot after Ch 6: helpers that later chapters extend (course plan 0.10).

Ch 6 adds make_worlds(xml, nworld). Ch 8 adds reset_done, nonfinite_reset, snapshot/restore;
Ch 9 adds describe_overflow; Ch 10 adds randomize; and so on.
"""
import mujoco
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw


def make_worlds(xml, nworld, nconmax=None, njmax=None, batch_sizes=None):
    """Compile an MJCF string and create nworld parallel worlds on the default device.

    Returns (mjm, m, d): the host mujoco.MjModel, the device mjw.Model and an mjw.Data whose
    arrays all start with an nworld dimension. nconmax is per world; njmax is per world.
    batch_sizes (e.g. {"body_mass": nworld}) gives chosen Model fields their own rows.
    """
    mjm = mujoco.MjModel.from_xml_string(xml)
    m = mjw.put_model(mjm, batch_sizes=batch_sizes)
    d = mjw.make_data(mjm, nworld=nworld, nconmax=nconmax, njmax=njmax)
    assert d.qpos.shape == (nworld, mjm.nq), d.qpos.shape
    return mjm, m, d

In [ ]:
# Example 6.9 - ex11_docstring_specs.py
"""Example 6.9 - read the shape rules from the library itself.

Two sources describe every field: the class docstring (human-readable, NumPy-style shapes
with the vector tail, e.g. "(nworld, nbody, 3)") and the declared spec on the dataclass
field (types.py::array, e.g. ('nworld', 'nbody') with dtype vec3). We print both next to
the shapes of a real Data/Model made by my_env.make_worlds.
Run: python ex11_docstring_specs.py   (my_env_step06.py must be next to it)
"""
import dataclasses
import os
import re
import sys

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

sys.path.insert(0, os.path.dirname(os.path.abspath(globals().get("__file__", "my_env.py"))))
sys.dont_write_bytecode = True                # no __pycache__ next to the examples
try:
    from my_env import make_worlds            # notebook / project copy
except ImportError:
    from my_env_step06 import make_worlds     # examples/06-batched-data-layout/

PLAYGROUND = """
<mujoco model="playground">
  <option timestep="0.01"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="overview" pos="0.5 -3 1.5" xyaxes="1 0 0 0 0.45 0.9"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/></body>
    <body name="box" pos="0.5 0 1"><freejoint/><geom type="box" size=".1 .1 .1" rgba=".3 .3 1 1"/></body>
    <body name="arm" pos="0 1 1"><joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .5 0 0" size=".03"/></body>
  </worldbody>
  <actuator><motor joint="hinge" gear="1"/></actuator>
  <sensor><jointpos joint="hinge"/></sensor>
  <keyframe><key name="up" qpos="0 0 2 1 0 0 0  0.5 0 2 1 0 0 0  0.3"/></keyframe>
</mujoco>
"""


def doc_shapes(cls):
    """Map field name -> shape string from the 'Attributes:' block of a class docstring."""
    out = {}
    for line in cls.__doc__.splitlines():
        hit = re.match(r"\s+(\w+): .*?(\([^()]*\))\s*$", line)
        if hit:
            out[hit.group(1)] = hit.group(2)
    return out


def spec(cls, name):
    """The declared dims of a dataclass field (the arguments of types.array)."""
    f = next(f for f in dataclasses.fields(cls) if f.name == name)
    return getattr(f.type, "shape", None), getattr(f.type, "dtype", None)


mjm, m, d = make_worlds(PLAYGROUND, nworld=8, nconmax=16, njmax=64)
print(f"make_worlds -> nworld={d.nworld}, nq={mjm.nq}, nbody={mjm.nbody}, naconmax={d.naconmax}")
print(f"{'field':<12}{'docstring':<24}{'spec':<30}{'warp':<9}{'numpy':<13}")
rows = [(mjw.Data, d, n) for n in ("qpos", "xpos", "xquat", "xmat", "sensordata", "nacon")]
rows += [(mjw.Contact, d.contact, n) for n in ("pos", "frame", "worldid")]
rows += [(mjw.Model, m, n) for n in ("body_mass", "geom_size", "geom_type")]
for cls, obj, name in rows:
    dims, dtype = spec(cls, name)
    arr = getattr(obj, name)
    spec_txt = f"({', '.join(map(str, dims))}) {dtype.__name__}"
    print(f"{name:<12}{doc_shapes(cls).get(name, '?'):<24}{spec_txt:<30}"
          f"{str(arr.shape):<9}{str(arr.numpy().shape):<13}")

## §10. What `.numpy()` costs
On a GPU every `.numpy()` is an allocation, a copy **and a sync**, on top of the sync an eager `mjw.step` already does inside the solver (`wp.capture_while`). On Warp's CPU device it is a view of the live buffer, so the example adds `.copy()`. Compare reading every step with reading once (each variant runs once untimed, then the best of 3 alternating runs is kept). On the CPU the gap is within noise; on Colab the copy + sync makes it real.

In [ ]:
# Example 6.10 - ex09_numpy_cost.py
"""Example 6.10 - what a .numpy() call costs inside a stepping loop.

arr.numpy() on a GPU array allocates host memory, copies device -> host and WAITS for every
queued kernel (a synchronization). An eager GPU mjw.step already syncs inside the solver
(wp.capture_while checks its counter on the host), so a read every step adds one more sync
plus the copy. We time NSTEP steps with a read every step against one read at the end.
Each variant runs once untimed first, then the two alternate for 3 repetitions and we keep the
minimum of each, so neither pays for being the first timed loop.
On the CPU device .numpy() is a view (no copy, no wait), so the gap is within timing noise;
on a GPU expect a larger gap (device -> host copy + sync), how large depends on the GPU and nworld.
Run: python ex09_numpy_cost.py
"""
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 64
NSTEP = 100

mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)
mjw.step(m, d)                     # warm up: JIT compile happens here, not in the timing
wp.synchronize()


def run(read_every_step):
    mjw.reset_data(m, d)
    wp.synchronize()
    t0 = time.perf_counter()
    zs = []
    for _ in range(NSTEP):
        mjw.step(m, d)
        if read_every_step:
            zs.append(d.qpos.numpy()[:, 2].copy())   # GPU: copy + sync; CPU: a view, so .copy()
    if not read_every_step:
        zs = d.qpos.numpy()[:, 2].copy()         # one read at the end
    wp.synchronize()
    return time.perf_counter() - t0, zs


run(False)                         # one untimed pass of each variant: the first timed loop
run(True)                          # after a fresh step is slower whatever it does
t_each, t_once = float("inf"), float("inf")
for _ in range(3):                 # alternate the order, keep the minimum of 3
    t_each = min(t_each, run(True)[0])
    t_once = min(t_once, run(False)[0])
dev = "GPU" if ON_GPU else "CPU dev box"
print(f"device: {wp.get_device()} ({dev}), nworld={NWORLD}, nstep={NSTEP}")
print(f"read every step: {1e3 * t_each / NSTEP:8.3f} ms/step")
print(f"read once      : {1e3 * t_once / NSTEP:8.3f} ms/step")
print(f"ratio          : {t_each / t_once:8.2f}x")

### §10 (cont.) Record on the device, read once, plot many worlds
A small kernel writes one row of a `(nstep, nworld)` buffer per step; one `.numpy()` at the end. The first cell passes `--dump-png .` so the example saves its two PNGs in the notebook's folder; the cell after it shows them.

In [ ]:
import sys
_argv = sys.argv
sys.argv = ["ex10_batched_plots.py", "--dump-png", "."]   # read by argparse in the next cell

In [ ]:
# Example 6.11 - ex10_batched_plots.py
"""Example 6.11 - many worlds at once: record on device, read once, then plot.

64 BALL_DROP worlds start at random heights; 256 PENDULUM worlds start at random q1.
A tiny kernel copies one column of the state into a (nstep, nworld) device buffer every
step, so the loop never syncs; one .numpy() at the end brings everything to the host.
Run: python ex10_batched_plots.py [--dump-json PATH] [--dump-png DIR]
  --dump-json PATH  writes PATH-stem_balls.json and _pendulums.json (decimated, for the page)
  --dump-png DIR    save the two matplotlib figures in DIR (default: no PNGs are written)
"""
import argparse
import json
import os

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
parser.add_argument("--dump-png", default=None)
args, _ = parser.parse_known_args()


@wp.kernel
def record(src: wp.array2d(dtype=float), col: int, step: int, out: wp.array2d(dtype=float)):
    worldid = wp.tid()
    out[step, worldid] = src[worldid, col]      # row = step, column = world


def simulate(xml, nworld, nstep, init, cols, nconmax=None, njmax=None):
    """Run nworld worlds; return {name: (nstep, nworld) host array} for qpos/qvel columns."""
    mjm = mujoco.MjModel.from_xml_string(xml)
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=nworld, nconmax=nconmax, njmax=njmax)
    qpos = np.tile(mjm.qpos0, (nworld, 1))
    init(qpos)
    d.qpos.assign(qpos)
    bufs = {name: wp.zeros((nstep, nworld), dtype=float) for name in cols}
    for i in range(nstep):
        mjw.step(m, d)
        for name, (field, col) in cols.items():
            wp.launch(record, dim=nworld, inputs=[getattr(d, field), col, i], outputs=[bufs[name]])
    out = {name: b.numpy() for name, b in bufs.items()}   # the only host reads
    out["overflow"] = d.overflow.numpy()
    return mjm, out


rng = np.random.default_rng(0)

# --- 64 bouncing balls ------------------------------------------------------------
NB, NSTEP_B = 64, 400
h0 = rng.uniform(0.4, 1.6, NB)


def init_ball(q):
    q[:, 2] = h0


mjm_b, ball = simulate(BALL_DROP, NB, NSTEP_B, init_ball, {"z": ("qpos", 2), "vz": ("qvel", 2)},
                       nconmax=4, njmax=16)
dt_b = mjm_b.opt.timestep
first_up = np.argmax(ball["vz"] > 0, axis=0)          # first step with upward velocity
t_bounce = (first_up + 1) * dt_b
t_pred = np.sqrt(2 * (h0 - 0.1) / 9.81)               # free fall to contact (radius 0.1)
print(f"balls: nworld={NB}, {NSTEP_B} steps of {dt_b} s, all bounced: {bool((first_up > 0).all())}")
print(f"  mean bounce time {t_bounce.mean():.4f} s, free-fall prediction {t_pred.mean():.4f} s")
print(f"  max |measured - predicted| {np.abs(t_bounce - t_pred).max():.4f} s")
print("  overflow:", "none" if not ball["overflow"].any() else ball["overflow"])

# --- 256 double pendulums -----------------------------------------------------------
NP, NSTEP_P = 256, 300
q10 = rng.uniform(-2.5, 2.5, NP)


def init_pend(q):
    q[:, 0] = q10


mjm_p, pend = simulate(PENDULUM, NP, NSTEP_P, init_pend, {"q1": ("qpos", 0), "v1": ("qvel", 0)})
flipped = (np.abs(pend["q1"]) > np.pi).any(axis=0)
print(f"pendulums: nworld={NP}, {NSTEP_P} steps of {mjm_p.opt.timestep} s")
print(f"  |v1| max over all worlds {np.abs(pend['v1']).max():.2f} rad/s,"
      f" worlds where link1 went over the top: {int(flipped.sum())}")

if args.dump_json:
    db, dp = 8, 6                                       # decimation (steps per frame)
    stem = os.path.splitext(args.dump_json)[0]          # data/ex10.json -> data/ex10_*.json
    balls = {"nworld": NB, "dt": db * dt_b, "kind": "height", "zmin": 0.0, "zmax": 1.7,
             "frames": np.round(ball["z"][::db].astype(np.float64), 3).tolist()}
    pends = {"nworld": NP, "dt": dp * mjm_p.opt.timestep, "kind": "phase",
             "q1": np.round(pend["q1"][::dp].astype(np.float64), 2).tolist(),
             "v1": np.round(pend["v1"][::dp].astype(np.float64), 2).tolist()}
    for suffix, payload in (("_balls.json", balls), ("_pendulums.json", pends)):
        with open(stem + suffix, "w") as f:
            json.dump(payload, f, separators=(",", ":"))
        print("wrote", stem + suffix)

if not args.dump_png:
    print("no plots saved (pass --dump-png DIR to write the two PNGs)")
else:
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        print("skipped plots: matplotlib not installed")
    else:
        t = np.arange(NSTEP_B) * dt_b
        fig, axes = plt.subplots(8, 8, figsize=(10, 8), sharex=True, sharey=True)
        for w, ax in enumerate(axes.flat):
            ax.plot(t, ball["z"][:, w], lw=1, color=f"C{w % 8}")
            ax.text(0.95, 0.9, f"w{w}", transform=ax.transAxes, ha="right", va="top", fontsize=7)
        fig.suptitle("ball z(t) [m] vs t [s], 64 worlds")
        fig.savefig(os.path.join(args.dump_png, "ex10_small_multiples.png"), dpi=80)
        fig, ax = plt.subplots(figsize=(6, 5))
        for w in range(NP):
            ax.plot(pend["q1"][:, w], pend["v1"][:, w], lw=0.4, alpha=0.5, color=f"C{w % 8}")
        ax.set_xlabel("q1 [rad]")
        ax.set_ylabel("v1 [rad/s]")
        ax.set_title("double pendulum phase portrait, 256 worlds")
        fig.savefig(os.path.join(args.dump_png, "ex10_phase_portrait.png"), dpi=80)
        print("saved ex10_small_multiples.png and ex10_phase_portrait.png in", args.dump_png)

In [ ]:
from IPython.display import Image, display
import os
sys.argv = _argv   # restore the kernel's own arguments
for name in ("ex10_small_multiples.png", "ex10_phase_portrait.png"):
    if os.path.exists(name):
        display(Image(name))
    else:
        print("not found (matplotlib missing?):", name)

## §11. When is the GPU faster? Throughput vs nworld
On Colab this runs nworld = 1, 16, 256, 4096. Copy the JSON line printed by the cell after it into the chapter page's throughput widget to plot your own curve. Expect one world to be slower than C MuJoCo (about 95 kernel launches of several µs each against about 3 µs for a whole C step, plus a host sync at each solver loop check when stepping eagerly); the first row (nworld = 1) of the table is your check. Many worlds win.

In [ ]:
# Example 6.12 - ex12_throughput_vs_nworld.py
"""Example 6.12 - when is MJWarp faster than C MuJoCo? World-steps per second vs nworld.

Eager stepping (no CUDA graph) of BALL_DROP for several nworld, next to C MuJoCo's
single-world mj_step. Each mjw.step is a sequence of kernel launches whose fixed cost does
not depend on nworld, so throughput grows with nworld until the device is full.
CPU numbers are from Warp's CPU device ("on a CPU dev box"), NOT GPU performance.
Run: python ex12_throughput_vs_nworld.py [--dump-json PATH]
"""
import argparse
import json
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLDS = [1, 16, 256, 4096] if ON_GPU else [1, 4, 16, 64]
NSTEP = 200 if ON_GPU else 50

mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)

# --- C MuJoCo, one world, float64 on the host CPU -------------------------------------
mjd = mujoco.MjData(mjm)
n_c = 2000
t0 = time.perf_counter()
for _ in range(n_c):
    mujoco.mj_step(mjm, mjd)
c_rate = n_c / (time.perf_counter() - t0)

# --- count kernel launches in one mjw.step (wrap wp.launch for one call; demo only) ------
counts = {"n": 0}
real_launch, real_tiled = wp.launch, wp.launch_tiled


def counting(fn):
    def wrapper(*a, **k):
        counts["n"] += 1
        return fn(*a, **k)
    return wrapper


d = mjw.make_data(mjm, nworld=1, nconmax=4, njmax=16)
mjw.step(m, d)                                  # JIT compile outside the count
wp.launch, wp.launch_tiled = counting(real_launch), counting(real_tiled)
mjw.step(m, d)
wp.launch, wp.launch_tiled = real_launch, real_tiled
print(f"kernel launches per mjw.step (BALL_DROP): {counts['n']}")

# --- MJWarp, eager stepping -------------------------------------------------------------
def time_steps(d, nstep):
    wp.synchronize()
    t0 = time.perf_counter()
    for _ in range(nstep):
        mjw.step(m, d)
    wp.synchronize()                            # wait for the device before reading the clock
    return time.perf_counter() - t0


# one discarded timed pass: the first timed loop in a process is slower whatever nworld is
time_steps(mjw.make_data(mjm, nworld=NWORLDS[0], nconmax=4, njmax=16), NSTEP)

rows = []
for nworld in NWORLDS:
    d = mjw.make_data(mjm, nworld=nworld, nconmax=4, njmax=16)
    for _ in range(30):
        mjw.step(m, d)                          # warm up: compile is cached, caches get hot
    dt = time_steps(d, NSTEP)
    rows.append({"nworld": nworld, "step_ms": 1e3 * dt / NSTEP, "rate": nworld * NSTEP / dt})
    assert not d.overflow.numpy().any()

dev = f"{wp.get_device()} ({'GPU' if ON_GPU else 'CPU dev box, not GPU performance'})"
print(f"device: {dev}, nstep={NSTEP}, eager (no graph)")
print(f"C MuJoCo mj_step, 1 world: {c_rate:12,.0f} steps/s")
print(f"{'nworld':>7} {'ms/step':>9} {'world-steps/s':>15} {'vs C':>8}")
for r in rows:
    print(f"{r['nworld']:>7} {r['step_ms']:>9.3f} {r['rate']:>15,.0f} {r['rate'] / c_rate:>7.4f}x")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"device": str(wp.get_device()), "on_gpu": ON_GPU, "nstep": NSTEP,
                   "launches_per_step": counts["n"], "c_rate": round(c_rate, 1),
                   "rows": [{k: round(v, 4) for k, v in r.items()} for r in rows]}, f)
    print("wrote", args.dump_json)

In [ ]:
import json
# the variables come from the Example 6.12 cell above (same format as --dump-json)
rows_json = {"device": str(wp.get_device()), "on_gpu": ON_GPU, "nstep": NSTEP,
             "launches_per_step": counts["n"], "c_rate": round(c_rate, 1),
             "rows": [{k: round(v, 4) for k, v in r.items()} for r in rows]}
print(json.dumps(rows_json))   # paste this line into the page's throughput widget

## Exercise 6.1: predict six shapes
Fill in `pred` **before** running. The cell checks your answers against the real arrays.

In [ ]:
PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

pred = {
    "d.qpos": (),            # Warp shape
    "d.xpos.numpy()": (),    # NumPy view
    "d.sensordata": (),
    "m.body_mass": (),
    "m.geom_type": (),
    "d.contact.pos": (),     # default nconmax = 48 per world
}
mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=32)
actual = {"d.qpos": d.qpos.shape, "d.xpos.numpy()": d.xpos.numpy().shape,
          "d.sensordata": d.sensordata.shape, "m.body_mass": m.body_mass.shape,
          "m.geom_type": m.geom_type.shape, "d.contact.pos": d.contact.pos.shape}
wrong = [k for k in pred if pred[k] != actual[k]]
for k in wrong:
    print(f"{k}: you said {pred[k]}, it is {actual[k]}")
assert not wrong, f"{len(wrong)} of 6 shapes wrong"
print("all six shapes correct")

In [ ]:
#@title Exercise 6.1: solution (double-click to show the code)
pred = {"d.qpos": (32, 2), "d.xpos.numpy()": (32, 3, 3), "d.sensordata": (32, 5),
        "m.body_mass": (1, 3), "m.geom_type": (2,), "d.contact.pos": (1536,)}
# sensordata: jointpos 1 + jointvel 1 + framepos 3 = 5; contact pool: 48 * 32 = 1536
assert all(pred[k] == actual[k] for k in pred), actual
print("all six shapes correct")

## Exercise 6.2: per-world wind
Give 4 ball worlds the winds (+5,0,0), (-5,0,0), (0,+5,0), (0,0,0) m/s via `m.opt.wind`, step 100 times and assert each ball drifts with its wind. Note the `density` in the model: without it `put_model` sets `m.has_fluid = False` and the wind is never read.

In [ ]:
BALL_WIND = """
<mujoco model="ball_wind">
  <option timestep="0.002" density="1.2"/>
  <worldbody>
    <geom name="floor" type="plane" size="5 5 .1"/>
    <body name="ball" pos="0 0 1"><freejoint/><geom type="sphere" size=".1" mass="1"/></body>
  </worldbody>
</mujoco>
"""

# your code: put_model, make_data(nworld=4, nconmax=4, njmax=16), set m.opt.wind,
# step 100 times, then:
# x = d.qpos.numpy()[:, :2]
# assert x[0, 0] > 1e-3 and x[1, 0] < -1e-3 and x[2, 1] > 1e-3 and abs(x[3]).max() < 1e-6

In [ ]:
#@title Exercise 6.2: solution (double-click to show the code)
mjm = mujoco.MjModel.from_xml_string(BALL_WIND)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=4, nconmax=4, njmax=16)
wind = np.array([[5, 0, 0], [-5, 0, 0], [0, 5, 0], [0, 0, 0]], dtype=np.float32)
m.opt.wind = wp.array(wind, dtype=wp.vec3)   # B = 4 = nworld
for _ in range(100):
    mjw.step(m, d)
x = d.qpos.numpy()[:, :2]
assert x[0, 0] > 1e-3 and x[1, 0] < -1e-3 and x[2, 1] > 1e-3 and abs(x[3]).max() < 1e-6
print("drift per world:", x)

## Stretch: seed per-world qpos in a kernel
Write a kernel launched with `dim=(d.nworld, m.nq)` that copies `qpos0` (with `worldid % qpos0.shape[0]`) and sets one address from `wp.rand_init(seed, worldid)` / `wp.randf`. Same seed, same result; time it against NumPy + `assign` for nworld = 4096.

In [ ]:
# your kernel here

In [ ]:
#@title Stretch: solution (double-click to show the code)
@wp.kernel
def seed_qpos(seed: int, qpos0: wp.array2d(dtype=float), adr: int, lo: float, hi: float,
              qpos: wp.array2d(dtype=float)):
    worldid, i = wp.tid()
    qpos[worldid, i] = qpos0[worldid % qpos0.shape[0], i]
    if i == adr:
        state = wp.rand_init(seed, worldid)
        qpos[worldid, i] = lo + (hi - lo) * wp.randf(state)

mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=8)
wp.launch(seed_qpos, dim=(d.nworld, mjm.nq), inputs=[42, m.qpos0, 0, -1.0, 1.0], outputs=[d.qpos])
a = d.qpos.numpy().copy()
wp.launch(seed_qpos, dim=(d.nworld, mjm.nq), inputs=[42, m.qpos0, 0, -1.0, 1.0], outputs=[d.qpos])
assert np.array_equal(a, d.qpos.numpy())
print(a[:, 0])